# GROMACS MD Simulation on GPU (Free)

**Goal:** Complete 10 ns MD simulation with free GPU (10-15× faster than CPU)

**Current progress:** 170 ps / 10,000 ps (1.7%)

**Estimated time:** 8-12 hours on T4 GPU

---

## ⚠️ IMPORTANT: Enable GPU

Before running any cells:
1. **Runtime** → **Change runtime type**
2. **Hardware accelerator** → **GPU** (T4)
3. Click **Save**


## Step 1: Verify GPU is Available

In [ ]:
# Check GPU
!nvidia-smi
print("\n✅ If you see GPU details above, you're ready!")
print("❌ If you see 'command not found', enable GPU in Runtime settings")

## Step 2: Install GROMACS with GPU Support

**Time:** ~15-20 minutes

In [ ]:
%%time
# Install GROMACS (pre-compiled with CUDA)
!apt-get update -qq
!apt-get install -y gromacs gromacs-data

# Verify installation
!gmx --version | grep -E "VERSION|GPU"

print("\n✅ GROMACS installed!")

## Step 3: Mount Google Drive

**Action needed:** You'll be prompted to authorize. Click the link and grant access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("Now upload 'colab_restart.zip' to your Google Drive root folder")

## Step 4: Extract Checkpoint Files

**Before running:** Upload `colab_restart.zip` (16 MB) to your Google Drive root folder

In [ ]:
# Extract files
!cd /content && unzip -o /content/drive/MyDrive/colab_restart.zip

# Verify files
!ls -lh md.tpr md.cpt topol.top md.mdp

print("\n✅ Files extracted!")

## Step 5: Run MD Simulation with GPU

**Time:** ~8-12 hours for remaining 9,830 ps

**⚠️ Important:** This cell will run for hours. Keep the browser tab open or use Colab's background execution.

**Monitor:** You'll see progress updates every 10 ps:
```
Step 100000, time 200.0 ps
Performance: 28.5 ns/day
```

In [ ]:
%%time
# Run GROMACS with GPU acceleration
!gmx mdrun -v -deffnm md -cpi md.cpt -ntomp 2 -nb gpu -pme gpu -bonded gpu -update gpu

# Flags explained:
# -v: verbose output
# -deffnm md: default filename prefix
# -cpi md.cpt: continue from checkpoint
# -ntomp 2: use 2 CPU threads (Colab gives 2 vCPUs)
# -nb gpu: non-bonded interactions on GPU
# -pme gpu: electrostatics on GPU
# -bonded gpu: bonded interactions on GPU
# -update gpu: coordinate updates on GPU

print("\n✅ Simulation complete!")

## Step 6: Check Results

In [ ]:
# Check output files
!ls -lh md.xtc md.log md.edr md.cpt

# Check final step
!tail -50 md.log | grep "Step" | tail -1

# Check performance
!grep "Performance" md.log | tail -5

## Step 7: Save Results to Google Drive

**Time:** ~5-10 minutes

In [ ]:
%%time
# Compress results
!zip -r md_results.zip md.xtc md.log md.edr md.cpt

# Check size
!ls -lh md_results.zip

# Copy to Google Drive
!cp md_results.zip /content/drive/MyDrive/

print("\n✅ Results saved to Google Drive!")
print("Download 'md_results.zip' from your Google Drive")

## 🔄 If Session Times Out (>12 hours)

Colab free tier disconnects after 12 hours. If that happens:

### Save Checkpoint Before Timeout:

In [ ]:
# Run this if you need to stop early
# (Interrupt the running cell first with stop button)

# Save checkpoint to Drive
!cp md.cpt /content/drive/MyDrive/md_restart.cpt
!cp md.log /content/drive/MyDrive/
!cp md.xtc /content/drive/MyDrive/

print("✅ Checkpoint saved! Safe to disconnect.")

### Resume After Reconnect:

In [ ]:
# After reconnecting, run setup cells (1-4) again, then:

# Restore checkpoint
!cp /content/drive/MyDrive/md_restart.cpt ./md.cpt

# Resume simulation
!gmx mdrun -v -deffnm md -cpi md.cpt -ntomp 2 -nb gpu -pme gpu -bonded gpu -update gpu

---

## 📊 Expected Performance

| Phase | Time |
|-------|------|
| Setup (Steps 1-4) | 20-30 min |
| MD Simulation (Step 5) | 8-12 hours |
| Save Results (Step 7) | 5-10 min |
| **Total** | **9-13 hours** |

**vs local CPU:** 5 days → saved 4+ days! 🎉

---

## 🐛 Troubleshooting

### Error: "nvidia-smi not found"
- Enable GPU: Runtime → Change runtime type → GPU → Save

### Error: "gmx: command not found"
- Re-run Step 2 (install GROMACS)

### Error: "File not found: colab_restart.zip"
- Make sure you uploaded the file to Google Drive root folder
- Check the path: should be `/content/drive/MyDrive/colab_restart.zip`

### Simulation crashes
- Check log: `!tail -100 md.log`
- Try using previous checkpoint: `!cp md_prev.cpt md.cpt`

---

## ✅ Success Criteria

Simulation is complete when:
- Step reaches 5,000,000
- Time reaches 10,000 ps (10 ns)
- Log shows "Finished mdrun"
- md.xtc file is ~500-1000 MB

---

*Created: January 27, 2026*  
*Project: p53 StabiliMut Initiative 2*
